In [51]:
import pandas as pd
from pathlib import Path

# Go from notebooks/stageC back to the project folder
project = Path.cwd().parents[1]
data = project / "data"

agents = pd.read_csv(data / "agents.csv")
services = pd.read_csv(data / "services.csv")
transactions = pd.read_csv(data / "transactions.csv")

print("Agents:", agents.shape)
print("Services:", services.shape)
print("Transactions:", transactions.shape)

Agents: (1290, 8)
Services: (18, 4)
Transactions: (214816, 8)


In [52]:
# Clean transaction columns used for the groupby results

transactions["status"] = (
    transactions["status"]
    .astype(str)
    .str.strip()
    .str.upper()
)

transactions["service_code"] = (
    transactions["service_code"]
    .astype(str)
    .str.strip()
    .str.upper()
)

transactions["amount_xaf"] = pd.to_numeric(
    transactions["amount_xaf"]
    .astype(str)
    .str.replace(r"[^0-9.-]", "", regex=True),
    errors="coerce"
)

In [53]:
# Total transaction amount for each status
groupby_1 = (
    transactions
    .groupby("status")["amount_xaf"]
    .sum()
    .reset_index()
    .sort_values("status")
)

groupby_1

,status,amount_xaf
0,FAILED,1.739203e+09
1,REVERSED,1.747657e+09
2,SUCCESS,1.055508e+10


In [54]:
# Count how many transactions each service has
groupby_2 = (
    transactions
    .groupby("service_code")
    .size()
    .reset_index(name="transaction_count")
    .sort_values("service_code")
)

groupby_2

,service_code,transaction_count
0,AGRI,11989
1,AIR,12091
2,BILL,11684
3,DATA,12121
4,DEP,11883
5,INS,11917
6,INTL,11868
7,LOAN,11945
8,MER,12099
9,P2P,11910


In [55]:
# Clean agent type for GROUPBY 3

agents["agent_type"] = (
    agents["agent_type"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [56]:
# Count how many agents there are for each agent type
groupby_3 = (
    agents
    .groupby("agent_type")
    .size()
    .reset_index(name="agent_count")
    .sort_values("agent_type")
)

groupby_3

,agent_type,agent_count
0,MASTER,151
1,RETAIL,480
2,SUB,659


In [57]:
from pathlib import Path

results = Path.cwd().parents[1] / "reports" / "groupby_results"
results.mkdir(parents=True, exist_ok=True)

groupby_1.to_csv(results / "pandas_groupby_1.csv", index=False)
groupby_2.to_csv(results / "pandas_groupby_2.csv", index=False)
groupby_3.to_csv(results / "pandas_groupby_3.csv", index=False)

print("Pandas results saved.")

Pandas results saved.


In [58]:
from pathlib import Path
import pandas as pd

results = Path.cwd().parents[1] / "reports" / "groupby_results"

# Read the Pandas and SQL results
pandas_1 = pd.read_csv(results / "pandas_groupby_1.csv")
sql_1 = pd.read_csv(results / "sql_groupby_1.csv")

pandas_2 = pd.read_csv(results / "pandas_groupby_2.csv")
sql_2 = pd.read_csv(results / "sql_groupby_2.csv")

pandas_3 = pd.read_csv(results / "pandas_groupby_3.csv")
sql_3 = pd.read_csv(results / "sql_groupby_3.csv")

# Compare the results
print("Groupby 1:", pandas_1.equals(sql_1))
print("Groupby 2:", pandas_2.equals(sql_2))
print("Groupby 3:", pandas_3.equals(sql_3))

Groupby 1: False
Groupby 2: False
Groupby 3: False


In [ ]:
print("GROUPBY 1")
print("Pandas:")
print(pandas_1)
print("\nSQL:")
print(sql_1)

print("\nGROUPBY 2")
print("Pandas:")
print(pandas_2)
print("\nSQL:")
print(sql_2)

# print("\nGROUPBY 3")
# print("Pandas:")
# print(pandas_3)
# print("\nSQL:")
# print(sql_3)

Pandas:
     status    amount_xaf
0    FAILED  1.739203e+09
1  REVERSED  1.747657e+09
2   SUCCESS  1.055508e+10

SQL:
     status    amount_xaf
0    FAILED  1.322033e+09
1  REVERSED  1.314624e+09
2   SUCCESS  7.979608e+09

GROUPBY 2
Pandas:
   service_code  transaction_count
0          AGRI              11989
1           AIR              12091
2          BILL              11684
3          DATA              12121
4           DEP              11883
5           INS              11917
6          INTL              11868
7          LOAN              11945
8           MER              12099
9           P2P              11910
10          PEN              11839
11          SAL              12039
12          SAV              11787
13          SCH              12023
14          TAX              11909
15         TRAN              11845
16         UTIL              11893
17          WTH              11974

SQL:
   service_code  transaction_count
0          AGRI               8954
1           AIR   